In [1]:
import scarf

scarf.set_verbosity('WARNING')

scarf.cytebase.connect("scarf_docs").download_dataset(
    'tenx_5K_pbmc_rnaseq',
    destination='scarf_datasets',
    zarr=True,
)
ds = scarf.DataStore(
    'scarf_datasets/tenx_5K_pbmc_rnaseq/data.zarr',
    nthreads=4,
    min_features_per_cell=10,
)
ds.filter_cells(
    attrs=['RNA_nCounts', 'RNA_nFeatures'],
    highs=[15000, 4000],
    lows=[1000, 500],
    reset_previous=True,
)
if 'I__hvgs' not in ds.RNA.feats.columns:
    ds.mark_hvgs(min_cells=20, top_n=500, show_plot=False)

In [2]:
normalized = ds.run_normalization(feat_key='hvgs')
pca = ds.run_pca(normalized, dims=15)
init = ds.build_embedding_initialization(pca, n_centroids=100)
ann = ds.build_ann_index(pca)
neighbors = ds.query_neighbors(ann, k=11)
graph = ds.build_connectivity_map(neighbors)

state = ds.get_assay_state('RNA')
(
    state.normalized,
    state.reduction,
    state.embedding_initialization,
    state.connectivity_map,
)

(ArtifactRef(scope='assay', kind='normalized', artifact_id='5ec491f66754576b4d90607eb5cfeaaf7a4e54bf2f559d2a836dfd167530d2ab', assay='RNA'),
 ArtifactRef(scope='assay', kind='reduction', artifact_id='fb91fd0551ec9bc8b014e23b34f6ecf827781259b077d5dff531396c02c6375c', assay='RNA'),
 ArtifactRef(scope='assay', kind='embedding_initialization', artifact_id='f3c81406273d46fa354d1aeab3a36b65cb558de614c6635f32aa3ddb775a0bae', assay='RNA'),
 ArtifactRef(scope='assay', kind='connectivity_map', artifact_id='b0cee467c6413653dff42f6f99c5c7df6b2bd6f2102fbc926bd0e7c37d3fab4f', assay='RNA'))

In [3]:
ds.run_umap(n_epochs=100, parallel=True)
ds.run_leiden_clustering(resolution=0.5)